# Chapter 9: Decorators and Closures
*Fluent Python, 2nd Edition -- Luciano Ramalho*

This notebook distills the key ideas from Chapter 9 into runnable code you can experiment with.

**Concepts covered:**
1. Decorator fundamentals and import-time execution
2. Variable scope rules and closures
3. The nonlocal declaration
4. Implementing a well-behaved decorator
5. Memoization with functools.cache and lru_cache
6. Single dispatch generic functions
7. Parameterized decorators (function-based and class-based)

---
## 1. Decorator Fundamentals

A decorator is a callable that takes a function and returns a function (or other callable).
The `@decorator` syntax is pure syntactic sugar:

```python
@decorate
def target(): ...
# is equivalent to:
target = decorate(target)
```

**Key fact:** decorators run at **import time** (when the module loads), not when the decorated function is called.

In [ ]:
# A decorator that replaces the original function
def deco(func):
    def inner():
        print("running inner()")
    return inner

@deco
def target():
    print("running target()")

# target now points to inner, NOT the original target
target()
print(f"target.__name__ = {target.__name__}")

### Import-Time Execution: Registration Decorators

Registration decorators run immediately when the module is loaded.
They typically add functions to a registry and return them unchanged.

In [ ]:
# Registration decorator: runs at definition time, returns func unchanged
registry = []

def register(func):
    print(f"  [import-time] registering {func.__name__}")
    registry.append(func)
    return func  # return the SAME function, not a wrapper

@register
def f1():
    print("running f1()")

@register
def f2():
    print("running f2()")

def f3():
    print("running f3()")

print(f"\nregistry = {[f.__name__ for f in registry]}")
print("Notice: register() ran BEFORE this print, at decoration time.")

---
## 2. Variable Scope Rules and Closures

### Python's Scope Decision at Compile Time

Python decides at **compile time** whether a variable is local based on whether
it is assigned anywhere in the function body -- even if the assignment comes
after a read.

In [ ]:
# Scope surprise: assignment anywhere makes a variable local everywhere in that function
b = 6

def f1(a):
    """b is NOT assigned in this function, so Python looks it up globally."""
    print(f"f1: a={a}, b={b}")

f1(3)  # works fine: b=6 from global scope

def f2(a):
    """b IS assigned below, so Python treats it as local -- even before the assignment."""
    print(f"f2: a={a}")
    try:
        print(f"f2: b={b}")  # UnboundLocalError!
    except UnboundLocalError as e:
        print(f"f2: ERROR -> {e}")
    b = 9  # this line makes b local to the ENTIRE function

f2(3)

### Closures

A **closure** is a function that captures **free variables** from the enclosing scope.
The free variable bindings are stored in `__closure__` and listed in `__code__.co_freevars`.

The classic example: a running-average calculator.

In [ ]:
# Closure example: make_averager captures `series` as a free variable
def make_averager():
    series = []  # local to make_averager, but free in averager

    def averager(new_value):
        series.append(new_value)  # mutating a list -- no rebinding!
        total = sum(series)
        return total / len(series)

    return averager

avg = make_averager()
print(avg(10))   # 10.0
print(avg(11))   # 10.5
print(avg(12))   # 11.0

# Inspect the closure
print(f"\nFree variables: {avg.__code__.co_freevars}")
print(f"Closure cell contents: {avg.__closure__[0].cell_contents}")

---
## 3. The nonlocal Declaration

If you need to **rebind** (assign to) a free variable of immutable type
(int, str, tuple...), Python will treat it as local -- breaking the closure.

`nonlocal` tells Python: "this name belongs to the enclosing scope, not to me."

In [ ]:
# BROKEN: without nonlocal, count and total become local
def make_averager_broken():
    count = 0
    total = 0

    def averager(new_value):
        # count += 1  -->  count = count + 1  -->  assignment makes count local!
        try:
            count += 1
            total += new_value
            return total / count
        except UnboundLocalError as e:
            return f"ERROR: {e}"

    return averager

avg_broken = make_averager_broken()
print("Broken:", avg_broken(10))

In [ ]:
# FIXED: nonlocal lets us rebind immutable free variables
def make_averager():
    count = 0
    total = 0

    def averager(new_value):
        nonlocal count, total  # <-- the fix
        count += 1
        total += new_value
        return total / count

    return averager

avg = make_averager()
print(avg(10))   # 10.0
print(avg(11))   # 10.5
print(avg(12))   # 11.0

---
## 4. Implementing a Well-Behaved Decorator

A proper decorator should:
1. Define an inner wrapper that accepts `*args, **kwargs`
2. Call the original function (captured as a free variable)
3. Use `@functools.wraps(func)` to preserve the original's metadata

In [ ]:
import time
import functools

# A well-behaved clock decorator
def clock(func):
    @functools.wraps(func)  # copies __name__, __doc__, __module__, etc.
    def clocked(*args, **kwargs):
        t0 = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - t0
        name = func.__name__
        arg_lst = [repr(arg) for arg in args]
        arg_lst.extend(f"{k}={v!r}" for k, v in kwargs.items())
        arg_str = ", ".join(arg_lst)
        print(f"[{elapsed:0.8f}s] {name}({arg_str}) -> {result!r}")
        return result
    return clocked

@clock
def factorial(n):
    """Compute n! recursively."""
    return 1 if n < 2 else n * factorial(n - 1)

print("6! =", factorial(6))
print(f"\nfactorial.__name__ = {factorial.__name__}")   # 'factorial', not 'clocked'
print(f"factorial.__doc__  = {factorial.__doc__}")       # preserved

---
## 5. Memoization with functools.cache and lru_cache

`@functools.cache` (Python 3.9+) stores **all** results -- unbounded memory.
`@functools.lru_cache` evicts least-recently-used entries (default `maxsize=128`).

Both require all arguments to be **hashable** (they are used as dict keys).

In [ ]:
import functools

# Without caching: exponential time
call_count = 0

def fibonacci_slow(n):
    global call_count
    call_count += 1
    if n < 2:
        return n
    return fibonacci_slow(n - 2) + fibonacci_slow(n - 1)

call_count = 0
result = fibonacci_slow(20)
print(f"fibonacci_slow(20) = {result}  (calls: {call_count:,})")

In [ ]:
import functools

# With @cache: linear time, each n computed only once
@functools.cache
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 2) + fibonacci(n - 1)

result = fibonacci(100)
print(f"fibonacci(100) = {result}")
print(f"Cache info: {fibonacci.cache_info()}")

# Clean up for later cells
fibonacci.cache_clear()

In [ ]:
import functools

# lru_cache with bounded size
@functools.lru_cache(maxsize=16, typed=False)
def expensive_computation(x, y):
    """Simulate an expensive function."""
    return x ** y

for x in range(4):
    for y in range(4):
        expensive_computation(x, y)

print(f"Cache info: {expensive_computation.cache_info()}")
# hits: repeated calls found in cache; misses: first-time calls

---
## 6. Single Dispatch Generic Functions

`@functools.singledispatch` creates a **generic function** that dispatches
on the **type of its first argument**.

- Decorate the base function (handles `object`)
- Register specialized implementations with `@base.register`
- Use type hints on the first parameter to specify the dispatch type

In [ ]:
from functools import singledispatch
from collections import abc
import html
import numbers
import fractions
import decimal

@singledispatch
def htmlize(obj: object) -> str:
    """Base: HTML-escape the repr of any object."""
    content = html.escape(repr(obj))
    return f'<pre>{content}</pre>'

@htmlize.register
def _(text: str) -> str:
    """Strings: escape, convert newlines to <br/>, wrap in <p>."""
    content = html.escape(text).replace('\n', '<br/>\n')
    return f'<p>{content}</p>'

@htmlize.register
def _(n: numbers.Integral) -> str:
    """Integers: show decimal and hex."""
    return f'<pre>{n} (0x{n:x})</pre>'

@htmlize.register
def _(n: bool) -> str:
    """Booleans: plain repr (bool is subtype of int, but gets priority)."""
    return f'<pre>{n}</pre>'

@htmlize.register
def _(seq: abc.Sequence) -> str:
    """Sequences (but not str): HTML list with recursive htmlize."""
    inner = '</li>\n<li>'.join(htmlize(item) for item in seq)
    return '<ul>\n<li>' + inner + '</li>\n</ul>'

@htmlize.register(fractions.Fraction)
def _(x) -> str:
    frac = fractions.Fraction(x)
    return f'<pre>{frac.numerator}/{frac.denominator}</pre>'

@htmlize.register(decimal.Decimal)
@htmlize.register(float)
def _(x) -> str:
    frac = fractions.Fraction(x).limit_denominator()
    return f'<pre>{x} ({frac.numerator}/{frac.denominator})</pre>'

# Test it
print(htmlize({1, 2, 3}))           # set -> base handler
print(htmlize('Heimlich & Co.\n- a game'))  # str
print(htmlize(42))                   # int
print(htmlize(True))                 # bool (more specific than int)
print(htmlize(2/3))                  # float
print(htmlize(['alpha', 66, {3, 2, 1}]))  # list -> Sequence

---
## 7. Parameterized Decorators

A parameterized decorator is really a **decorator factory**: it takes
configuration arguments and returns the actual decorator.

Three layers of nesting:
```python
def factory(params):     # <-- factory, called with config
    def decorator(func): # <-- actual decorator
        def wrapper():   # <-- replacement function
            ...
        return wrapper
    return decorator
```

Usage: `@factory(params)` -- the `()` calls the factory, which returns the decorator.

In [ ]:
# Parameterized registration decorator
registry = set()

def register(active=True):
    """Decorator factory: returns a decorator that registers or unregisters."""
    def decorate(func):
        if active:
            registry.add(func)
        else:
            registry.discard(func)
        return func  # registration decorator returns func unchanged
    return decorate

@register(active=False)
def f1():
    print("running f1()")

@register()  # NOTE: must call with () even for defaults
def f2():
    print("running f2()")

def f3():
    print("running f3()")

print("Registry:", {f.__name__ for f in registry})  # only f2

# Register f3 at runtime
register()(f3)
print("After register()(f3):", {f.__name__ for f in registry})

# Unregister f2
register(active=False)(f2)
print("After unregistering f2:", {f.__name__ for f in registry})

In [ ]:
import time

# Parameterized clock decorator with custom format string
DEFAULT_FMT = '[{elapsed:0.8f}s] {name}({args}) -> {result}'

def clock(fmt=DEFAULT_FMT):
    """Decorator factory: accepts a format string."""
    def decorate(func):
        def clocked(*_args):
            t0 = time.perf_counter()
            _result = func(*_args)
            elapsed = time.perf_counter() - t0
            name = func.__name__
            args = ', '.join(repr(arg) for arg in _args)
            result = repr(_result)
            print(fmt.format(**locals()))
            return _result
        return clocked
    return decorate

@clock()  # default format
def snooze_default(seconds):
    time.sleep(seconds)

@clock('{name}: {elapsed:.4f}s')  # custom format
def snooze_custom(seconds):
    time.sleep(seconds)

snooze_default(0.05)
snooze_custom(0.05)

### Class-Based Parameterized Decorator

Using a class with `__init__` (for params) and `__call__` (as the decorator)
can be clearer than triple-nested functions.

In [ ]:
import time

DEFAULT_FMT = '[{elapsed:0.8f}s] {name}({args}) -> {result}'

class clock:
    """Parameterized clock decorator implemented as a class."""

    def __init__(self, fmt=DEFAULT_FMT):
        self.fmt = fmt

    def __call__(self, func):
        """Called when the decorator is applied to a function."""
        def clocked(*_args):
            t0 = time.perf_counter()
            _result = func(*_args)
            elapsed = time.perf_counter() - t0
            name = func.__name__
            args = ', '.join(repr(arg) for arg in _args)
            result = repr(_result)
            print(self.fmt.format(**locals()))
            return _result
        return clocked

@clock()  # default format
def snooze(seconds):
    time.sleep(seconds)

@clock('{name}({args}) dt={elapsed:0.3f}s')  # custom format
def snooze_short(seconds):
    time.sleep(seconds)

snooze(0.05)
snooze_short(0.05)

---
## Key Takeaways

| Concept | One-liner |
|---|---|
| Decorator syntax | `@dec` is sugar for `func = dec(func)` |
| Import-time execution | Decorators run when the module loads, not when the function is called |
| Variable scope | Assignment anywhere in a function makes a name local to the **entire** function |
| Closure | A function + bindings for its free variables, stored in `__closure__` |
| `nonlocal` | Lets an inner function rebind an immutable free variable from the enclosing scope |
| `@functools.wraps` | Copies `__name__`, `__doc__`, etc. from the wrapped function to the wrapper |
| `@functools.cache` | Unbounded memoization (Python 3.9+); all args must be hashable |
| `@functools.lru_cache` | Bounded memoization with `maxsize`; evicts least-recently-used entries |
| `@singledispatch` | Type-based dispatch on first argument; extensible via `@base.register` |
| Parameterized decorator | A factory that returns the real decorator: `@factory(args)` |
| Class-based decorator | `__init__` stores params, `__call__` is the decorator |